In [1]:
import importlib
from snowflake.snowpark.functions import col,trim,split,lit
from snowflake.snowpark.functions import col, sum as _sum, when, is_null
from snowflake.snowpark import functions as F
import sys 
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np

In [2]:
session = Session.builder.configs(snowflake_conn_prop).create()

In [3]:
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

### Read the data

In [4]:
data = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Modelling_after_updating_sales_numbers\features_not_collapsed\Input Data Preparation\Validated_data_for_TFT_training.csv",index_col=["MONTH_OF_SALE"],parse_dates=True)

### Step 4 - Creating the features 

In [5]:
#Step 1 - Since PITRU_PAKSH and PITRAPAKSHA_DAYS both are same, Will drop PITRAPAKSHA_DAYS column
data=data.drop(columns=["PITRAPAKSHA_DAYS"])

In [6]:
#Step 2 - Removing following columns 
#1. 'TOTAL_DAYS_FESTIVE_PHASE_I', 
#2. 'TOTAL_DAYS_FESTIVE_PHASE_II',
#3. 'TOTAL_DAYS_FESTIVE_PHASE_III', 
#4. 'TOTAL_DAYS_PITRU_PAKSH'
#Because these columns are either 0 or same for all the months

cols_to_drop = ['TOTAL_DAYS_FESTIVE_PHASE_I','TOTAL_DAYS_FESTIVE_PHASE_II','TOTAL_DAYS_FESTIVE_PHASE_III','TOTAL_DAYS_PITRU_PAKSH']

data=data.drop(columns=cols_to_drop)

In [7]:
#Step 3 - Remove following columns
#1. PROP_EVENT_PITRU_PAKSH 
#2. PROP_EVENT_FESTIVE_PHASE_I
#3. PROP_EVENT_FESTIVE_PHASE_II
#4. PROP_EVENT_FESTIVE_PHASE_III
#5. NUM_FESTIVE_DAYS_MONTH

cols_to_drop2 = ['PROP_EVENT_PITRU_PAKSH','PROP_EVENT_FESTIVE_PHASE_I','PROP_EVENT_FESTIVE_PHASE_II','PROP_EVENT_FESTIVE_PHASE_III','NUM_FESTIVE_DAYS_MONTH']
data=data.drop(columns=cols_to_drop2)

In [8]:
#Separate out the festival columns
# festive_cols = [i for i in df.columns if '_DAYS' in i and '_FESTIVE' not in i and i!='MARRIAGE_DAYS']
# festive_cols

In [9]:
# #Step 4 - Create one single columns to capture the number of FESTIVAL_DAYS
# df.loc[:,"TOTAL_FESTIVAL_DAYS"] = df[festive_cols].sum(axis=1)

In [10]:
# #Step 5 - Drop all the festive_cols
# df = df.drop(columns=festive_cols)

### Drop the festive features columns

In [11]:
cols_to_drop3 = ['NAVRATRI_DAYS',
 'DUSSEHRA_(VIJAYADASHAMI)_DAYS',
 'KARWA_CHAUTH_DAYS',
 'DHANTERAS_DAYS',
 'DIWALI_DAYS',
 'GOVARDHAN_POOJA_DAYS',
 'BHAI_DOOJ_DAYS',
 'CHHATH_PUJA_DAYS']

data.drop(columns=cols_to_drop3,inplace=True)

In [27]:
#Merge with Diwali dataframe
diwali_df = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\6_month_horizon\Updated_diwali_features\Input_Data_Preparation\Dataframe_with_D_plus_minus_features.csv",index_col=["MONTH_OF_SALE"],parse_dates=True,date_format='%d-%m-%Y')

In [28]:
diwali_df

,D-30,D-29,D-28,D-27,D-26,D-25,D-24,D-23,D-22,D-21,...,D-2,D-1,D0,D+1,D+2,D+3,D+4,D+5,D+6,D+7
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,
2023-10-01,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2023-11-01,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
2024-10-01,1,1,1,1,1,1,1,1,1,1,...,1,1,1,0,0,0,0,0,0,0
2024-11-01,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,1,1,1,1,1
2025-09-01,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2025-10-01,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
2026-10-01,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2026-11-01,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1


In [17]:
list_of_columns = ['AKSHAYA_TRITIYA_DAYS', 'BUDDHA_PURNIMA_DAYS', 'EID_UL_FITR_DAYS',
       'GANESH_CHATURTHI_DAYS', 'GANGA_DUSSEHRA_DAYS', 'GURU_PURNIMA_DAYS',
       'HANUMAN_JAYANTI_DAYS', 'HARTALIK_TEEJ_DAYS', 'HOLI_DAYS',
       'HOLIKA_DAHAN_DAYS', 'JAGANNATH_RATHYATRA_DAYS', 'JANMASHTAMI_DAYS',
       'LOHRI_DAYS', 'MAHA_SHIVARATRI_DAYS', 'MAKAR_SANKRANTI_PONGAL_DAYS',
       'NAG_PANCHAMI_DAYS', 'NEW_YEAR_DAYS', 'ONAM_DAYS',
       'RAKSHA_BANDHAN_DAYS', 'REPUBLIC_DAY_DAYS', 'VASANT_PANCHAMI_DAYS',
       'VISHWAKARMA_PUJA_DAYS', 'MARRIAGE_DAYS', 'FESTIVE_PHASE_I',
       'FESTIVE_PHASE_II', 'FESTIVE_PHASE_III', 'PITRU_PAKSH',
       'PROP_FESTIVE_PHASE_I', 'PROP_FESTIVE_PHASE_II',
       'PROP_FESTIVE_PHASE_III', 'PROP_PITRU_PAKSH','D-30', 'D-29', 'D-28', 'D-27',
       'D-26', 'D-25', 'D-24', 'D-23', 'D-22', 'D-21', 'D-20', 'D-19', 'D-18',
       'D-17', 'D-16', 'D-15', 'D-14', 'D-13', 'D-12', 'D-11', 'D-10', 'D-9',
       'D-8', 'D-7', 'D-6', 'D-5', 'D-4', 'D-3', 'D-2', 'D-1', 'D0', 'D+1',
       'D+2', 'D+3', 'D+4', 'D+5', 'D+6', 'D+7']

In [20]:
diwali_cols = ['D-30', 'D-29', 'D-28', 'D-27',
       'D-26', 'D-25', 'D-24', 'D-23', 'D-22', 'D-21', 'D-20', 'D-19', 'D-18',
       'D-17', 'D-16', 'D-15', 'D-14', 'D-13', 'D-12', 'D-11', 'D-10', 'D-9',
       'D-8', 'D-7', 'D-6', 'D-5', 'D-4', 'D-3', 'D-2', 'D-1', 'D0', 'D+1',
       'D+2', 'D+3', 'D+4', 'D+5', 'D+6', 'D+7']

In [23]:
data.index.dtype

dtype('<M8[ns]')

In [26]:
diwali_df.head()

,D-30,D-29,D-28,D-27,D-26,D-25,D-24,D-23,D-22,D-21,...,D-2,D-1,D0,D+1,D+2,D+3,D+4,D+5,D+6,D+7
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,
2023-01-10,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2023-01-11,0,0,0,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,1
2024-01-10,1,1,1,1,1,1,1,1,1,1,...,1,1,1,0,0,0,0,0,0,0
2024-01-11,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,1,1,1,1,1,1
2025-01-09,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [30]:
df = data.join(diwali_df,how='left')

In [31]:
df.loc['2026-10-01',diwali_cols].drop_duplicates().T

MONTH_OF_SALE,2026-10-01
D-30,1.0
D-29,1.0
D-28,1.0
D-27,1.0
D-26,1.0
D-25,1.0
D-24,1.0
D-23,1.0
D-22,1.0
D-21,1.0


In [32]:
df.fillna(value=0,inplace=True)

In [33]:

#Step 6 - Festive intensity columns
#First create FY columns 
fy_23_date_mask = ((df.index>='2022-04-01') & (df.index<='2023-03-31'))
fy_24_date_mask = ((df.index>='2023-04-01') & (df.index<='2024-03-31'))
fy_25_date_mask = ((df.index>='2024-04-01') & (df.index<='2025-03-31'))
fy_26_date_mask = ((df.index>='2025-04-01') & (df.index<='2026-03-31'))
fy_27_date_mask = ((df.index>='2026-04-01') & (df.index<='2027-03-31'))

df.loc[fy_23_date_mask,'FINANCIAL_YEAR'] = "FY'23"

df.loc[fy_24_date_mask,'FINANCIAL_YEAR'] = "FY'24"

df.loc[fy_25_date_mask,'FINANCIAL_YEAR'] = "FY'25"

df.loc[fy_26_date_mask,'FINANCIAL_YEAR'] = "FY'26"

df.loc[fy_27_date_mask,'FINANCIAL_YEAR'] = "FY'27"



In [34]:
df["TOTAL_MONTHLY_SALES"] = df.groupby(df.index.to_period('M'))["NET_SALES"].transform('sum')

In [35]:
df["MONTH_OF_SALE_COLUMN"] = df.index.strftime("%B - %Y")

In [36]:
df["TOTAL_YEARLY_SALES"] = df.groupby("FINANCIAL_YEAR")["NET_SALES"].transform('sum')

In [37]:
monthly_agg = df.loc[:,['MONTH_OF_SALE_COLUMN','TOTAL_MONTHLY_SALES',"TOTAL_YEARLY_SALES"]]
monthly_agg.drop_duplicates(inplace=True)

In [38]:
monthly_agg.sort_index(inplace=True)

In [39]:
# monthly_agg["DEVIATION"] = ((monthly_agg["TOTAL_MONTHLY_SALES"] - monthly_agg["AVERAGE_YEARLY_SALES"])/monthly_agg["AVERAGE_YEARLY_SALES"])*100

In [40]:
monthly_agg

,MONTH_OF_SALE_COLUMN,TOTAL_MONTHLY_SALES,TOTAL_YEARLY_SALES
MONTH_OF_SALE,,,
2022-04-01,April - 2022,438531.0,4687708.0
2022-05-01,May - 2022,490765.0,4687708.0
2022-06-01,June - 2022,370919.0,4687708.0
2022-07-01,July - 2022,283284.0,4687708.0
2022-08-01,August - 2022,289076.0,4687708.0
...,...,...,...
2026-12-01,December - 2026,0.0,1340977.0
2027-01-01,January - 2027,0.0,1340977.0
2027-02-01,February - 2027,0.0,1340977.0


In [41]:
monthly_agg.loc[((monthly_agg.index>='2022-04-01') & (monthly_agg.index<='2023-03-31')),:]["TOTAL_MONTHLY_SALES"].sum()

4687708.0

In [42]:
monthly_agg["MONTHLY_CONTRIBUTION"] = np.round((monthly_agg["TOTAL_MONTHLY_SALES"]/monthly_agg["TOTAL_YEARLY_SALES"]),4)

In [43]:
# monthly_agg.loc[monthly_agg["MONTH_OF_SALE_COLUMN"].str.contains('April'),:]
monthly_agg.to_csv(r"Aggregate_results.csv")

### Last year contribution is according to what happened last year

In [44]:
monthly_agg["LAST_YEAR_CONTRIBUTION"] = monthly_agg.MONTHLY_CONTRIBUTION.shift(periods=12)

In [45]:
final_data=monthly_agg.loc[((monthly_agg.index>='2023-04-01') & (monthly_agg.index<='2026-11-01')),:]

In [46]:
final_data.index

DatetimeIndex(['2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01',
               '2023-08-01', '2023-09-01', '2023-10-01', '2023-11-01',
               '2023-12-01', '2024-01-01', '2024-02-01', '2024-03-01',
               '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01',
               '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01',
               '2024-12-01', '2025-01-01', '2025-02-01', '2025-03-01',
               '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01',
               '2025-08-01', '2025-09-01', '2025-10-01', '2025-11-01',
               '2025-12-01', '2026-01-01', '2026-02-01', '2026-03-01',
               '2026-04-01', '2026-05-01', '2026-06-01', '2026-07-01',
               '2026-08-01', '2026-09-01', '2026-10-01', '2026-11-01'],
              dtype='datetime64[ns]', name='MONTH_OF_SALE', freq=None)

In [47]:
df.head()

,PARENT_DEALER_CODE,MODEL_FAMILY,PARENT_DEALER_CODE_MODEL_FAMILY,BRAKE_TYPE,IGNITION_TYPE,WHEEL_TYPE,COLOUR,NET_SALES,AKSHAYA_TRITIYA_DAYS,BUDDHA_PURNIMA_DAYS,...,D+2,D+3,D+4,D+5,D+6,D+7,FINANCIAL_YEAR,TOTAL_MONTHLY_SALES,MONTH_OF_SALE_COLUMN,TOTAL_YEARLY_SALES
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,
2026-08-01,10910,GLAMOUR,10910<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK AND AC...,DRUM,SELF,CAST,BLACK AND ACCENT,0.0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,FY'27,0.0,August - 2026,1340977.0
2025-02-01,10910,GLAMOUR,10910<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK AND AC...,DRUM,SELF,CAST,BLACK AND ACCENT,0.0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,FY'25,356003.0,February - 2025,5256582.0
2023-11-01,10910,GLAMOUR,10910<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK AND AC...,DRUM,SELF,CAST,BLACK AND ACCENT,3.0,0,0,...,1.0,1.0,1.0,1.0,1.0,1.0,FY'24,1033206.0,November - 2023,5136645.0
2023-06-01,10910,GLAMOUR,10910<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK AND AC...,DRUM,SELF,CAST,BLACK AND ACCENT,0.0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,FY'24,390658.0,June - 2023,5136645.0
2022-11-01,10910,GLAMOUR,10910<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK AND AC...,DRUM,SELF,CAST,BLACK AND ACCENT,0.0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,FY'23,265457.0,November - 2022,4687708.0


In [48]:
df.columns

Index(['PARENT_DEALER_CODE', 'MODEL_FAMILY', 'PARENT_DEALER_CODE_MODEL_FAMILY',
       'BRAKE_TYPE', 'IGNITION_TYPE', 'WHEEL_TYPE', 'COLOUR', 'NET_SALES',
       'AKSHAYA_TRITIYA_DAYS', 'BUDDHA_PURNIMA_DAYS', 'EID_UL_FITR_DAYS',
       'GANESH_CHATURTHI_DAYS', 'GANGA_DUSSEHRA_DAYS', 'GURU_PURNIMA_DAYS',
       'HANUMAN_JAYANTI_DAYS', 'HARTALIK_TEEJ_DAYS', 'HOLI_DAYS',
       'HOLIKA_DAHAN_DAYS', 'JAGANNATH_RATHYATRA_DAYS', 'JANMASHTAMI_DAYS',
       'LOHRI_DAYS', 'MAHA_SHIVARATRI_DAYS', 'MAKAR_SANKRANTI_PONGAL_DAYS',
       'NAG_PANCHAMI_DAYS', 'NEW_YEAR_DAYS', 'ONAM_DAYS',
       'RAKSHA_BANDHAN_DAYS', 'REPUBLIC_DAY_DAYS', 'VASANT_PANCHAMI_DAYS',
       'VISHWAKARMA_PUJA_DAYS', 'MARRIAGE_DAYS', 'FESTIVE_PHASE_I',
       'FESTIVE_PHASE_II', 'FESTIVE_PHASE_III', 'PITRU_PAKSH',
       'PROP_FESTIVE_PHASE_I', 'PROP_FESTIVE_PHASE_II',
       'PROP_FESTIVE_PHASE_III', 'PROP_PITRU_PAKSH', 'DEALER_CITY',
       'X_CITY_CATEGORY', 'ZONAL_OFFICE_NAME', 'D-30', 'D-29', 'D-28', 'D-27',
       'D-

In [49]:
df.drop(columns=['FINANCIAL_YEAR', 'TOTAL_MONTHLY_SALES','TOTAL_YEARLY_SALES'],inplace=True)

In [50]:
monthly_agg.columns

Index(['MONTH_OF_SALE_COLUMN', 'TOTAL_MONTHLY_SALES', 'TOTAL_YEARLY_SALES',
       'MONTHLY_CONTRIBUTION', 'LAST_YEAR_CONTRIBUTION'],
      dtype='object')

In [51]:
monthly_agg = monthly_agg.loc[:,['MONTH_OF_SALE_COLUMN','LAST_YEAR_CONTRIBUTION']]

In [52]:
#Join the two dataframes 
merged_df = pd.merge(left=df,right=monthly_agg,on=["MONTH_OF_SALE_COLUMN"])

In [53]:
df.shape

(7726321, 81)

In [54]:
merged_df.rename(columns={'MONTH_OF_SALE_COLUMN':'MONTH_OF_SALE'},inplace=True)

In [55]:
merged_df["MONTH_OF_SALE"] = pd.to_datetime(merged_df["MONTH_OF_SALE"],format="%B - %Y")
merged_df.set_index("MONTH_OF_SALE",inplace=True)

In [56]:
merged_df.isnull().sum()

PARENT_DEALER_CODE                       0
MODEL_FAMILY                             0
PARENT_DEALER_CODE_MODEL_FAMILY          0
BRAKE_TYPE                               0
IGNITION_TYPE                            0
                                    ...   
D+4                                      0
D+5                                      0
D+6                                      0
D+7                                      0
LAST_YEAR_CONTRIBUTION             1519932
Length: 81, dtype: int64

### Data filtration

In [57]:
merged_df.sort_index(inplace=True)

In [58]:
merged_df_new = merged_df.loc['2023-04-01':'2026-11-01',:]
merged_df_new.index

DatetimeIndex(['2023-04-01', '2023-04-01', '2023-04-01', '2023-04-01',
               '2023-04-01', '2023-04-01', '2023-04-01', '2023-04-01',
               '2023-04-01', '2023-04-01',
               ...
               '2026-11-01', '2026-11-01', '2026-11-01', '2026-11-01',
               '2026-11-01', '2026-11-01', '2026-11-01', '2026-11-01',
               '2026-11-01', '2026-11-01'],
              dtype='datetime64[ns]', name='MONTH_OF_SALE', length=5573084, freq=None)

In [59]:
null_value_cols = merged_df_new.isnull().sum().index[merged_df_new.isnull().sum()!=0]
null_value_cols

Index([], dtype='object')

In [60]:
# 1. Calculate the basic aggregations first (Vectorized = Fast)
span_counts = merged_df_new.groupby('PARENT_DEALER_CODE_MODEL_FAMILY')['NET_SALES'].agg(
    min_date=lambda x: x.index.min(),
    max_date=lambda x: x.index.max(),
    count='count',
    min_net_sales='min',
    max_net_sales='max',
    q1_NET_SALES=lambda x: x.quantile(0.25),
    q2_NET_SALES=lambda x: x.quantile(0.50),
    q3_NET_SALES=lambda x: x.quantile(0.75),
    std_dev='std',
    mean_sales='mean',
    non_zero_count=lambda x: (x > 0).sum()
).reset_index()

# 2. Add the calculated columns (Time span and Outlier logic)
span_counts['time_span_in_months'] = (
    (span_counts['max_date'].dt.year - span_counts['min_date'].dt.year) * 12 + 
    (span_counts['max_date'].dt.month - span_counts['min_date'].dt.month)
)

#2a. Add coefficient of variation
span_counts["Coefficient_of_variation"] = span_counts['std_dev']/span_counts['mean_sales']
span_counts['percent_of_non_zero_sales'] = span_counts['non_zero_count'] / span_counts['count']

# 3. Corrected Outlier Logic (Checking if max is 5x greater than mean)
span_counts['outlier_sales'] = (span_counts['max_net_sales'] / span_counts['mean_sales'] > 5).map(
    {True: 'outliers present', False: 'outliers not present'}
)

In [61]:
# Filter 1: Keep only series with a minimum percentage of active months
# 0.30 means the series must have active sales in at least ~30% of the months 
min_activity_threshold = 0.30  

# Filter 2: The 18-month history requirement we calculated for Train + Val + Output chunks
min_historical_months = 18    

# Apply the filters to get valid combination names
valid_combinations = span_counts[
    (span_counts['percent_of_non_zero_sales'] >= min_activity_threshold) & 
    (span_counts['count'] >= min_historical_months)
]['PARENT_DEALER_CODE_MODEL_FAMILY'].unique()

# Filter your main dataframe using the whitelisted combinations
df_filtered = merged_df[merged_df['PARENT_DEALER_CODE_MODEL_FAMILY'].isin(valid_combinations)]

print(f"Remaining active series for TFT training: {len(valid_combinations)}")

print(f"Filtered data shape is : {df_filtered.shape}")

print(f"Full data shape is : {df.shape}")

Remaining active series for TFT training: 36098
Filtered data shape is : (2201978, 81)
Full data shape is : (7726321, 81)


In [62]:
df_filtered_copy = df_filtered.copy()

### Step 3 - Saving the valid series combinations in a snowflake table

In [63]:
valid_combination_df = pd.DataFrame({"VALID_PARENT_DEALER_CODE_MODEL_FAMILY_COMBINATIONS":valid_combinations})

In [64]:
valid_combination_df.shape

(36098, 1)

In [65]:
valid_combination_snowpark_df = session.create_dataframe(valid_combination_df)

In [66]:
valid_combination_snowpark_df.write.mode('overwrite').save_as_table('MOP_DATABASE.SOQ.VALID_COMBINATIONS_USED_FOR_TFT')

In [67]:
series_not_included = set(df["PARENT_DEALER_CODE_MODEL_FAMILY"]) - set(df_filtered["PARENT_DEALER_CODE_MODEL_FAMILY"])
len(series_not_included)

90563

In [68]:
series_not_included_df = pd.DataFrame({"SERIES_NAME":list(series_not_included)})

In [69]:
series_not_included_snowpark_df = session.create_dataframe(series_not_included_df)

In [70]:
series_not_included_snowpark_df.write.mode('overwrite').save_as_table('MOP_DATABASE.SOQ.INVALID_PARENT_DEALER_COMBINATIONS')

### Save series A and series B data

In [71]:
series_not_included_list = list(series_not_included)

In [72]:
series_A_data = merged_df_new.loc[merged_df_new["PARENT_DEALER_CODE_MODEL_FAMILY"].isin(valid_combinations),:]
series_B_data = merged_df_new.loc[merged_df_new["PARENT_DEALER_CODE_MODEL_FAMILY"].isin(series_not_included),:]

In [73]:
print(f"Shape of series A data is {series_A_data.shape}")
print(f"Shape of series B data is {series_B_data.shape}")

Shape of series A data is (1588312, 81)
Shape of series B data is (3984772, 81)


In [74]:
series_A_data.columns

Index(['PARENT_DEALER_CODE', 'MODEL_FAMILY', 'PARENT_DEALER_CODE_MODEL_FAMILY',
       'BRAKE_TYPE', 'IGNITION_TYPE', 'WHEEL_TYPE', 'COLOUR', 'NET_SALES',
       'AKSHAYA_TRITIYA_DAYS', 'BUDDHA_PURNIMA_DAYS', 'EID_UL_FITR_DAYS',
       'GANESH_CHATURTHI_DAYS', 'GANGA_DUSSEHRA_DAYS', 'GURU_PURNIMA_DAYS',
       'HANUMAN_JAYANTI_DAYS', 'HARTALIK_TEEJ_DAYS', 'HOLI_DAYS',
       'HOLIKA_DAHAN_DAYS', 'JAGANNATH_RATHYATRA_DAYS', 'JANMASHTAMI_DAYS',
       'LOHRI_DAYS', 'MAHA_SHIVARATRI_DAYS', 'MAKAR_SANKRANTI_PONGAL_DAYS',
       'NAG_PANCHAMI_DAYS', 'NEW_YEAR_DAYS', 'ONAM_DAYS',
       'RAKSHA_BANDHAN_DAYS', 'REPUBLIC_DAY_DAYS', 'VASANT_PANCHAMI_DAYS',
       'VISHWAKARMA_PUJA_DAYS', 'MARRIAGE_DAYS', 'FESTIVE_PHASE_I',
       'FESTIVE_PHASE_II', 'FESTIVE_PHASE_III', 'PITRU_PAKSH',
       'PROP_FESTIVE_PHASE_I', 'PROP_FESTIVE_PHASE_II',
       'PROP_FESTIVE_PHASE_III', 'PROP_PITRU_PAKSH', 'DEALER_CITY',
       'X_CITY_CATEGORY', 'ZONAL_OFFICE_NAME', 'D-30', 'D-29', 'D-28', 'D-27',
       'D-

In [75]:
series_A_data.to_csv(r"Series_A_data.csv")

series_B_data.to_csv(r"Series_B_data.csv")